# 04. Create Chroma DB (TL_내과)

이 노트북은 아래 2개 스크립트를 순서대로 실행합니다.
1. `src/transform_tl_internal_docs.py`
2. `src/load_chunks_to_chroma.py`


## 0) 경로 설정

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
INPUT_JSON = PROJECT_ROOT / 'data/processed/chroma_documents/TL_내과_통합_documents.json'
CHUNKS_JSON = PROJECT_ROOT / 'data/processed/chroma_documents/TL_내과_통합_chunks.json'
PERSIST_DIR = PROJECT_ROOT / 'data/processed/chroma_db/TL_내과_통합'
COLLECTION_NAME = 'TL_내과_통합'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('INPUT_JSON   =', INPUT_JSON)
print('CHUNKS_JSON  =', CHUNKS_JSON)
print('PERSIST_DIR  =', PERSIST_DIR)


## 1) 입력 파일 존재 확인

In [ ]:
assert INPUT_JSON.exists(), f'입력 파일이 없습니다: {INPUT_JSON}'
print('OK:', INPUT_JSON)


## 2) `transform_tl_internal_docs.py` 실행

출력: `TL_내과_통합_chunks.json`


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'src/transform_tl_internal_docs.py'),
    '--input', str(INPUT_JSON),
    '--output', str(CHUNKS_JSON),
]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## 3) 변환 결과 간단 검증

In [ ]:
import json

with CHUNKS_JSON.open('r', encoding='utf-8') as f:
    chunks = json.load(f)

print('records =', len(chunks))
print('sample doc_id =', chunks[0]['doc_id'])
print('sample keys =', list(chunks[0].keys()))

# 예시 확인 (qa_id=1235)
target = next((x for x in chunks if x.get('qa_id') == 1235), None)
if target:
    print('\nqa_id=1235 doc_id:', target['doc_id'])
    print('disease_names:', target['disease_names'])
    print('aliases:', target['aliases'])


## 4) `chromadb` 설치 확인

`chromadb`가 없으면 아래 셀을 실행하세요.


In [ ]:
import importlib.util

has_chromadb = importlib.util.find_spec('chromadb') is not None
print('chromadb installed:', has_chromadb)


In [ ]:
# 필요할 때만 실행
# %pip install chromadb


## 5) `load_chunks_to_chroma.py` 실행

persist directory를 만들고 collection에 upsert 합니다.


In [ ]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'src/load_chunks_to_chroma.py'),
    '--input', str(CHUNKS_JSON),
    '--persist-dir', str(PERSIST_DIR),
    '--collection-name', COLLECTION_NAME,
    '--text-field', 'retrieval_text',
    '--reset-collection',
]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## 6) 적재 결과 확인

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=str(PERSIST_DIR))
col = client.get_collection(COLLECTION_NAME)
print('persist_dir:', PERSIST_DIR)
print('collection:', COLLECTION_NAME)
print('count:', col.count())

sample = col.get(limit=1)
print('sample ids:', sample.get('ids'))
